In [106]:
import os, sys
sys.path.append("../..")
from utilities import plot_wellbore, RogiDataset
from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm
from plotly import express as px
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

data_path = "../../rogii-wellbore-geology-prediction"

# TVT Prediction Architecture

## Problem Framing

The task is a **signal registration problem**. At each measured depth (MD) step along a lateral wellbore, we want to find where in the stratigraphic column the well sits — expressed as a TVT value from the typewell reference log.

The primary signal is the GR log measured along the lateral, which must be registered against the typewell GR log indexed by TVT. The typewell acts as a stratigraphic reference: it tells you what GR you should expect at each TVT position. Finding the TVT at each MD step is therefore equivalent to finding where in the typewell GR profile the lateral GR matches best.

## Target Variable

The model outputs a probability distribution (softmax) over TVT candidates at each MD step. The TVT candidates are not arbitrary bins — they are a direct lookup into the typewell, centered on the local TVT anchor:

$$\mathcal{T} = \{TVT_{\text{anchor}} - 50, \; TVT_{\text{anchor}} - 49.5, \; \ldots, \; TVT_{\text{anchor}} + 49.5, \; TVT_{\text{anchor}} + 50\}$$

Since the typewell is sampled at 0.5 TVT resolution, this gives exactly **200 candidates** per window. The candidates are a slice of the typewell array, so each candidate directly corresponds to a real typewell GR value — no discretization decisions required.

The model is trained with cross-entropy loss against the ground truth TVT bin index at each MD step. At inference, the argmax over the 200 candidates gives the predicted TVT, which is converted back to an absolute TVT value for RMSE scoring.

## Sliding Window Training

Rather than one pass per well covering the full lateral, the model is trained on sliding windows along the MD axis. For each window:

- The **local anchor** is the last known TVT just before the window starts.
- The **typewell slice** of 200 candidates is centered on that local anchor TVT.
- The **target** is delta TVT relative to the local anchor at each MD step within the window.

This means every window position in every well is a valid training example, not just the prediction zone — significantly multiplying the effective dataset size and ensuring the model sees consistent input structure regardless of where in the lateral the window falls.

At inference, multiple overlapping windows are run across the lateral, each producing TVT estimates for its region. Overlapping predictions are averaged, acting as test-time augmentation and smoothing out individual window errors.

## Architecture

The model has three components: two 1D CNN heads and a 2D CNN fusion network.

### Typewell Head

A 1D CNN that takes the 200-candidate typewell GR slice as input and outputs a dense feature map of shape $(C, 200)$. Each position encodes the local GR pattern around that TVT candidate in a learned feature space.

### Lateral Head

A 1D CNN with the same structure but separate weights, taking the GR window as input and outputting a feature map of shape $(C, N_{md})$. Each position encodes the local GR pattern around that MD step.

### Outer Product Grid

The two feature maps are combined into a 2D grid of shape $(2C, 200, N_{md})$ by broadcasting and concatenating:

$$\text{grid}[:, t, d] = [f_{\text{tw}}[:, t] \;||\; f_{\text{lat}}[:, d]]$$

Every cell $(t, d)$ holds the concatenated features for TVT candidate $t$ and MD step $d$ simultaneously. This exposes the full comparison between every TVT candidate and every MD position to the downstream network — the registration problem is laid out spatially in the grid.

### Fusion CNN

A 2D CNN that takes the $(2C, 200, N_{md})$ grid and outputs a $(1, 200, N_{md})$ score map. A softmax over the 200 TVT candidates at each MD step gives the final probability distribution. The 2D CNN learns to find the smooth path of high similarity through the grid — equivalent to learned dynamic time warping.

At 200 TVT candidates × $N_{md}$ window steps, the grid is small and memory-tractable, allowing generous batch sizes and feature channels.

## Preprocessing

- NaNs in the lateral GR are dropped and the log is resampled onto a uniform MD grid via linear interpolation.
- Both logs are normalized per well to zero mean and unit variance before input.
- The typewell GR slice of 200 candidates is extracted centered on the local anchor TVT for each window.

## Data Augmentation

- **Sliding windows**: every window position along every well is a training example, each with its own local anchor. This is the primary source of data augmentation.
- **Vertical shifts**: shift the local anchor TVT by a random constant offset, moving the typewell slice up or down. The GR signal is unchanged but the registration target shifts.
- **Stretching and compression**: warp the typewell TVT axis locally, simulating formation thickness variation between the typewell and the lateral. This teaches the model robustness to imperfect typewell-to-lateral correspondence.

,X,Y,Z,ANCC,ASTNU,ASTNL,EGFDU,EGFDL,BUDA,TVT,GR,TVT_input,Known
MD,,,,,,,,,,,,,
11959.0,2944745.19,1015034.18,-9911.74,-10348.30,-10553.78,-10566.57,-10588.57,-10633.35,-10805.72,11987.96,NaN,11987.96,True
11960.0,2944745.19,1015034.17,-9912.74,-10348.30,-10553.78,-10566.57,-10588.57,-10633.35,-10805.72,11988.96,90.721937,11988.96,True
11961.0,2944745.19,1015034.16,-9913.74,-10348.30,-10553.78,-10566.57,-10588.57,-10633.35,-10805.72,11989.96,87.443718,11989.96,True
11962.0,2944745.18,1015034.15,-9914.74,-10348.30,-10553.78,-10566.57,-10588.57,-10633.35,-10805.72,11990.96,86.421577,11990.96,True
11963.0,2944745.18,1015034.14,-9915.74,-10348.30,-10553.78,-10566.57,-10588.57,-10633.35,-10805.72,11991.96,88.005436,11991.96,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
18375.0,2946229.19,1009297.03,-10925.86,-10597.04,-10802.52,-10815.31,-10837.31,-10882.08,-11054.46,12753.34,NaN,NaN,False
18376.0,2946229.32,1009296.04,-10925.89,-10597.08,-10802.56,-10815.35,-10837.35,-10882.12,-11054.50,12753.32,106.136931,NaN,False
18377.0,2946229.46,1009295.04,-10925.91,-10597.12,-10802.60,-10815.39,-10837.39,-10882.16,-11054.54,12753.31,106.422394,NaN,False


In [72]:
class RogiDataset():
    def __init__(self, path=Path("../../rogii-wellbore-geology-prediction/train",)):
        self.path = path
        self.well_files={
            f.name.replace("__horizontal_well.csv",""):{"Well":f,"TypeWell":f.parent/f.name.replace("__horizontal_well.csv","__typewell.csv")}
            for f in self.path.glob("*__horizontal_well.csv")
            }
        self.keys = list(self.well_files.keys())
        self._index=0

    def __getitem__(self,key:str|int):
        if isinstance(key, int):
            key = self.keys[key]
        well_data, typewell_data = pd.read_csv(self.well_files[key]["Well"]),pd.read_csv(self.well_files[key]["TypeWell"])
        well_data, typewell_data = well_data.set_index("MD").sort_index(), typewell_data.set_index("TVT").sort_index()
        pred_start_idx = well_data.index[(~well_data.TVT_input.isna()).sum()]
        last_known_idx = well_data.index[~well_data.TVT_input.isna()][-1]
        well_data["Known"] = well_data.index<=last_known_idx
        return well_data, typewell_data

    def __len__(self):
        return len(self.keys)
    
    def __iter__(self):
        self._index=0
        return self

    def __next__(self):
        if self._index>= len(self):
            raise StopIteration
        val = self[self._index]
        self._index+=1
        return val
        

In [73]:
self = RogiDataset()


In [ ]:
delta_tvts = {}
for w,(well_data, typewell_data) in enumerate(tqdm(self)):
    delta_tvts[w] = (well_data[~well_data.Known].TVT.agg(["max","min"]) - well_data[well_data.Known].TVT_input.iloc[-1]).values
delta_tvts = pd.DataFrame(delta_tvts, index=["Max","Min"]).T
q = 0.025
delta_tvt_bounds = (delta_tvts.Min.quantile(q), delta_tvts.Max.quantile(1-q))

  0%|          | 0/773 [00:00<?, ?it/s]

In [135]:
delta_tvts = []
for w,(well_data, typewell_data) in enumerate(tqdm(self)):
    delta_tvts.append((well_data[~well_data.Known].TVT) - well_data[well_data.Known].TVT_input.iloc[-1])
    break

  0%|          | 0/773 [00:00<?, ?it/s]

In [136]:
well_data[well_data.Known].TVT_input.iloc[-1]

np.float64(11747.37)

In [137]:
typewell_data.loc[well_data[well_data.Known].TVT_input.iloc[-1]-50:well_data[well_data.Known].TVT_input.iloc[-1]+50]

,GR,Geology
TVT,,
11697.45,70.48,EGFDL
11697.95,69.52,EGFDL
11698.45,70.92,EGFDL
11698.95,76.05,EGFDL
11699.45,83.36,EGFDL
...,...,...
11794.95,57.87,LBHL
11795.45,59.36,LBHL
11795.95,67.47,LBHL


In [130]:
px.histogram(delta_tvts[1])

In [124]:
delta_tvt_bounds

(np.float64(-47.54999999999891), np.float64(50.38500000000107))

In [113]:
np.log(well_data[~well_data.Known].TVT.agg(["max","min"])/well_data[well_data.Known].TVT_input.iloc[-1])

max    0.000743
min   -0.000956
Name: TVT, dtype: float64

In [ ]:
px.histogram(np.log(delta_tvts_bounds[delta_tvts_bounds.Max>0].Max))

In [94]:
(well_data[~well_data.Known].TVT.agg(["max","min"]) - well_data[well_data.Known].TVT_input.iloc[-1]).values

array([ 14.32, -11.08])

In [ ]:
well_data[well_data.Known].TVT_input.iloc[-1]

np.float64(11747.37)

max     2.66
min   -12.54
Name: TVT, dtype: float64

In [ ]:
class GRHead(nn.Module):
    """1D CNN to encode a GR log into a feature map."""

    def __init__(self, in_channels=1, feature_channels=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(in_channels, 16, kernel_size=7, padding=3),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Conv1d(16, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Conv1d(32, feature_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(feature_channels),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.encoder(x)


class FusionCNN(nn.Module):
    """2D CNN over the outer product grid (2C, N_tvt, N_md) -> (1, N_tvt, N_md)."""

    def __init__(self, in_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Conv2d(16, 1, kernel_size=1),
        )

    def forward(self, x):
        # x: (batch, 2C, N_tvt, N_md)
        return self.net(x).squeeze(1)  # (batch, N_tvt, N_md)


class TVTModel(nn.Module):
    """
    Two-head CNN for TVT registration.

    Typewell head encodes GR indexed by TVT.
    Lateral head encodes GR indexed by MD.
    Outer product constructs a (2C, N_tvt, N_md) cost volume.
    2D CNN fusion outputs a softmax probability over TVT at each MD step.

    Args:
        feature_channels: output channels from each 1D CNN head.
        downsample_md: factor to downsample the lateral GR before encoding.
        downsample_tvt: factor to downsample the typewell GR before encoding.
    """

    def __init__(self, feature_channels=32, downsample_md=10, downsample_tvt=1):
        super().__init__()
        self.downsample_md = downsample_md
        self.downsample_tvt = downsample_tvt

        self.typewell_head = GRHead(in_channels=1, feature_channels=feature_channels)
        self.lateral_head = GRHead(in_channels=1, feature_channels=feature_channels)
        self.fusion = FusionCNN(in_channels=feature_channels * 2)

    def forward(self, typewell_gr, lateral_gr):
        """
        Args:
            typewell_gr: (batch, 1, N_tvt) — GR indexed by TVT
            lateral_gr:  (batch, 1, N_md)  — GR indexed by MD

        Returns:
            probs: (batch, N_tvt_ds, N_md_ds) — softmax over TVT at each MD step
        """
        # Downsample along sequence dims to reduce cost volume size
        if self.downsample_tvt > 1:
            typewell_gr = F.avg_pool1d(typewell_gr, self.downsample_tvt)
        if self.downsample_md > 1:
            lateral_gr = F.avg_pool1d(lateral_gr, self.downsample_md)

        tw_feat = self.typewell_head(typewell_gr)  # (batch, C, N_tvt_ds)
        lat_feat = self.lateral_head(lateral_gr)   # (batch, C, N_md_ds)

        N_tvt = tw_feat.shape[2]
        N_md = lat_feat.shape[2]

        # Construct outer product grid: (batch, 2C, N_tvt, N_md)
        tw_grid = tw_feat.unsqueeze(3).expand(-1, -1, -1, N_md)
        lat_grid = lat_feat.unsqueeze(2).expand(-1, -1, N_tvt, -1)
        grid = torch.cat([tw_grid, lat_grid], dim=1)

        logits = self.fusion(grid)             # (batch, N_tvt_ds, N_md_ds)
        probs = F.softmax(logits, dim=1)       # softmax over TVT dim
        return probs


def tvt_loss(probs, tvt_labels):
    """
    Cross entropy loss over TVT distribution at each MD step.

    Args:
        probs:      (batch, N_tvt, N_md) — softmax probabilities
        tvt_labels: (batch, N_md)        — integer TVT bin index per MD step

    Returns:
        scalar loss
    """
    batch, N_tvt, N_md = probs.shape
    # (batch, N_md, N_tvt) -> (batch*N_md, N_tvt)
    logits_flat = probs.permute(0, 2, 1).reshape(-1, N_tvt)
    labels_flat = tvt_labels.reshape(-1)
    return F.cross_entropy(logits_flat, labels_flat)


if __name__ == "__main__":
    batch_size = 2
    N_tvt = 400      # raw TVT steps
    N_md = 7000      # raw MD steps

    model = TVTModel(feature_channels=32, downsample_md=10, downsample_tvt=1)

    typewell_gr = torch.randn(batch_size, 1, N_tvt)
    lateral_gr = torch.randn(batch_size, 1, N_md)

    probs = model(typewell_gr, lateral_gr)

    N_tvt_ds = N_tvt            # no tvt downsampling in this example
    N_md_ds = N_md // 10        # 700 after 10x downsample
    print(f"Output shape: {probs.shape}")  # (2, 400, 700)

    # Fake labels: integer TVT bin at each downsampled MD step
    tvt_labels = torch.randint(0, N_tvt_ds, (batch_size, N_md_ds))
    loss = tvt_loss(probs, tvt_labels)
    print(f"Loss: {loss.item():.4f}")

    # Expected TVT at each MD step (for RMSE evaluation)
    # Convert bin index back to TVT value using your TVT bin edges
    tvt_pred_bins = probs.argmax(dim=1)  # (batch, N_md_ds)
    print(f"Predicted TVT bin shape: {tvt_pred_bins.shape}")